# Explainability, re-run on the leakage-guarded pipeline

**Answers editor comments 7 and 8.** Nothing here reuses the submitted analysis: the
design matrix is rebuilt from the published predictor allowlist, the split is grouped on
school, and every explainer setting the editor asked for is recorded and written out.

What this produces, in `results/explainability/`:

| Artefact | Editor comment |
|---|---|
| `global_shap_<task>.csv` — mean absolute SHAP per feature, bootstrap CI | 7 (global vs. local kept separate) |
| `local_shap_<task>.csv` — quantile-stratified instances, not argmax/argmin | 7 (instance selection rule stated) |
| `lime_stability_<task>.csv` — 30 seeds per instance, weight SD, rank correlation | 7 (explanation stability) |
| `explainer_config.csv` — background, output scale, settings, per explainer | 7 (settings reported) |
| `codebook_resolved.csv` — official item wording from the SPSS metadata | 7 (documented item descriptions) |
| `fig_*.pdf` — vector figures for the manuscript | 10 |

**Runtime: about 35 minutes** on a Colab CPU runtime, High-RAM not required if
`analytic.parquet` exists. Cells are independent and checkpoint to Drive, so a
disconnect costs you one cell, not the run.

**Set `RUN_ALL_TASKS = False`** (cell 1) to do Low vs. High only in ~12 minutes.

---

> **A caution that belongs at the top.** A file in the working repository reports a column
> named `mean_abs_shap` containing negative values. A mean of absolute values cannot be
> negative; that file is permutation importance under a misleading name and is not SHAP.
> Do not merge it with anything produced here.


## Cell 1 — Setup

Mounts Drive, installs the two explainability libraries, finds the data.

In [ ]:

RUN_ALL_TASKS = True      # False -> Low vs. High only (~12 min)
PLAUSIBLE_VALUE = 1       # explanations are reported at a single PV; stated in the paper
RANDOM_SEED     = 20180101

PRIMARY_COUNTRY      = "ESP"   # the manuscript's analytic sample is Spain only
MAX_MISSING_COLUMNS  = 400     # the manuscript's exclusion rule
EXPECTED_N           = 29786   # what the two filters above must reproduce
EXPECTED_SCHOOLS     = 1087

N_GLOBAL_SHAP   = 2000    # stratified test instances for the global aggregation
N_SHAP_BOOTSTRAP= 1000    # resamples for the CI on mean |SHAP|
N_LOCAL_SHAP    = 6       # quantile-stratified instances for local explanation
N_LIME_SEEDS    = 30      # repeats per instance, for the stability analysis
N_LIME_SAMPLES  = 5000    # perturbations per LIME explanation

import os, sys, json, time, warnings, subprocess, hashlib
warnings.filterwarnings("ignore")

T0 = time.time()
def tick(msg):
    print(f"[{(time.time()-T0)/60:6.2f} min] {msg}", flush=True)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = "/content/drive/MyDrive/An_Explainable_AI_Education"
else:
    ROOT = os.environ.get("VLPSO_ROOT", ".")

for pkg, imp in [("shap", "shap"), ("lime", "lime"), ("pyreadstat", "pyreadstat")]:
    try:
        __import__(imp)
    except ImportError:
        tick(f"installing {pkg}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np, pandas as pd
import shap, lime, lime.lime_tabular
import sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

OUT = os.path.join(ROOT, "results", "explainability")
os.makedirs(OUT, exist_ok=True)

PARQUET = os.path.join(ROOT, "data", "processed", "analytic.parquet")

# The .sav is not always at data/raw/ -- notebook 00 unzips it into data/raw/STU/.
# Search for it rather than assuming, because a miss here silently costs the codebook
# and every item gloss with it.
def find_sav(root, name="CY07_MSU_STU_QQQ.sav"):
    for cand in (os.path.join(root, "data", "raw", name),
                 os.path.join(root, "data", "raw", "STU", name)):
        if os.path.exists(cand):
            return cand
    for dirpath, _, files in os.walk(os.path.join(root, "data")):
        if name in files:
            return os.path.join(dirpath, name)
    return os.path.join(root, "data", "raw", name)      # canonical path for the error message

SAV = find_sav(ROOT)
for p in (PARQUET, SAV):
    tick(f"{'FOUND  ' if os.path.exists(p) else 'missing'}  {p}")

assert os.path.exists(PARQUET) or os.path.exists(SAV), (
    "Neither the processed parquet nor the raw .sav is present. Run notebook 00 first, "
    "or set VLPSO_ROOT to the folder that holds data/."
)

def _ver(name):
    """lime ships no __version__ attribute; ask the installed-package metadata."""
    try:
        from importlib.metadata import version
        return version(name)
    except Exception:
        return "unknown"

ENV = {
    "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__, "shap": shap.__version__, "lime": _ver("lime"),
    "seed": RANDOM_SEED, "plausible_value": PLAUSIBLE_VALUE,
}
print(json.dumps(ENV, indent=1))
tick("Cell 1 OK")


## Cell 2 — Allowlist, leakage guard, design matrix

The allowlist is the 31 student-questionnaire items of Supplementary Table S1, reproduced
here so the notebook stands alone. The guard **raises**; it does not warn. That distinction
is the whole point — the submitted pipeline's dtype filter failed silently.

`assert_no_leakage` is called at every point a design matrix is built, and a deliberate
negative test at the end of the cell proves the guard fires.

In [ ]:

ALLOWLIST = [
    # ST011 — home educational resources
    "ST011Q01TA","ST011Q02TA","ST011Q03TA","ST011Q04TA","ST011Q05TA","ST011Q06TA",
    "ST011Q07TA","ST011Q08TA","ST011Q09TA","ST011Q10TA","ST011Q11TA","ST011Q12TA",
    "ST011Q16NA",
    # ST012 — household possessions, counts
    "ST012Q01TA","ST012Q02TA","ST012Q03TA","ST012Q05NA","ST012Q06NA","ST012Q07NA",
    "ST012Q08NA","ST012Q09NA",
    # ST013 — books in the home
    "ST013Q01TA",
    # ST019 — country of birth, student and parents
    "ST019AQ01T","ST019BQ01T","ST019CQ01T",
    # ST166 — digital-safety situational judgement (phishing e-mail)
    "ST166Q01HA","ST166Q02HA","ST166Q03HA","ST166Q04HA","ST166Q05HA",
    # demographic
    "ST004D01T",
]
assert len(ALLOWLIST) == 31 and len(set(ALLOWLIST)) == 31

GROUP_COL, WEIGHT_COL = "CNTSCHID", "W_FSTUWT"
CUT_L3, CUT_L5 = 482.38, 606.99          # official PISA 2018 boundaries, half-open below

import re
FORBIDDEN = [
    re.compile(r"^PV\d+(MATH|READ|SCIE)", re.I),
    re.compile(r"^math[_ ]?(score|category)$", re.I),
    re.compile(r"^label$", re.I),
    re.compile(r"^W_FST(UWT|URWT\d+)$", re.I),
    re.compile(r"^SENWT$", re.I),
    re.compile(r"^STRATUM", re.I),
    re.compile(r"^(CNTSCHID|CNTSTUID|CNT|CYC|NatCen|SUBNATIO|OECD)$", re.I),
    re.compile(r"noise_control", re.I),
]

class LeakageError(RuntimeError):
    pass

def assert_no_leakage(columns, where=""):
    """Raise on any column that is the outcome, derived from it, or a design variable.

    Matches on the stem before a one-hot or missing-indicator suffix, so
    STRATUM_ESP9033 and missingindicator_math_score are caught as well.
    """
    bad = []
    for c in columns:
        stem = re.sub(r"^missingindicator_", "", str(c).strip())
        for pat in FORBIDDEN:
            if pat.search(stem) or pat.search(stem.split("_")[0]):
                bad.append(c); break
    if bad:
        raise LeakageError(f"forbidden column(s) in design matrix at {where}: {sorted(set(bad))[:12]}")
    return True

# ---- load -----------------------------------------------------------------
if os.path.exists(PARQUET):
    df = pd.read_parquet(PARQUET)
    tick(f"parquet loaded: {df.shape[0]:,} rows x {df.shape[1]:,} cols")
else:
    import pyreadstat
    keep = ALLOWLIST + [GROUP_COL, WEIGHT_COL, "CNT"] + [f"PV{i}MATH" for i in range(1, 11)]
    df, meta = pyreadstat.read_sav(SAV, usecols=keep)
    df = df[df["CNT"] == "ESP"].copy()
    tick(f"sav loaded and filtered to ESP: {df.shape[0]:,} rows")

missing = [c for c in ALLOWLIST + [GROUP_COL, WEIGHT_COL] if c not in df.columns]
assert not missing, f"columns absent from the data: {missing}"

# ---- restrict to the analytic frame ---------------------------------------
# analytic.parquet is the MERGED PRE-EXCLUSION frame (Spain + Portugal, 41,875 records),
# not the analytic sample. Fitting on it silently explains a different population from the
# one the paper reports, and inflates the fold AUC because the extra records are easier.
# The two exclusions the manuscript specifies are therefore applied here explicitly.
n_before = len(df)
if "CNT" in df.columns:
    df = df[df["CNT"] == PRIMARY_COUNTRY].copy()
    tick(f"country filter CNT == '{PRIMARY_COUNTRY}': {n_before:,} -> {len(df):,}")
else:
    tick("WARNING: no CNT column; cannot verify the country restriction")

n_country = len(df)
miss_per_row = df.isna().sum(axis=1)
df = df[miss_per_row <= MAX_MISSING_COLUMNS].copy()
tick(f"missingness filter (<= {MAX_MISSING_COLUMNS} of {df.shape[1]:,} columns): "
     f"{n_country:,} -> {len(df):,} students in {df[GROUP_COL].nunique():,} schools")

if (len(df), df[GROUP_COL].nunique()) != (EXPECTED_N, EXPECTED_SCHOOLS):
    print("\n" + "!" * 78)
    print(f"ANALYTIC FRAME MISMATCH: got {len(df):,} students in "
          f"{df[GROUP_COL].nunique():,} schools; the manuscript reports "
          f"{EXPECTED_N:,} in {EXPECTED_SCHOOLS:,}.")
    print("Do NOT report these attributions as coming from the analytic sample until this")
    print("is reconciled. The likely cause is a different missingness basis: the manuscript")
    print("counts missing values over the 1,119 columns of the student questionnaire file,")
    print("so a merged frame with school-questionnaire columns will not reproduce it.")
    print("!" * 78 + "\n")
else:
    tick(f"analytic frame matches the manuscript: {EXPECTED_N:,} students, "
         f"{EXPECTED_SCHOOLS:,} schools")

pv_col = f"PV{PLAUSIBLE_VALUE}MATH"
assert pv_col in df.columns, f"{pv_col} absent; cannot derive the proficiency category"

# ---- outcome, derived per plausible value ---------------------------------
def category_from(score):
    return pd.cut(score, bins=[-np.inf, CUT_L3, CUT_L5, np.inf], labels=["Low","Medium","High"])

df["_cat"] = category_from(df[pv_col])
tick("proficiency distribution (unweighted): " +
     df["_cat"].value_counts(normalize=True).round(4).to_dict().__str__())

# ---- design matrix --------------------------------------------------------
X_all = df[ALLOWLIST].copy()
ind   = X_all.isna().astype(int).add_prefix("missingindicator_")
ind   = ind.loc[:, ind.sum() > 0]                       # drop all-observed indicators
X_all = pd.concat([X_all.fillna(X_all.median(numeric_only=True)), ind], axis=1)

assert_no_leakage(X_all.columns, where="design matrix")
tick(f"design matrix: {X_all.shape[1]} columns "
     f"({len(ALLOWLIST)} items + {ind.shape[1]} missingness indicators)")

groups  = df[GROUP_COL].to_numpy()
weights = df[WEIGHT_COL].to_numpy()

# ---- negative test: the guard must fire -----------------------------------
try:
    assert_no_leakage(list(X_all.columns) + ["PV1MATH"], where="negative test")
    raise AssertionError("GUARD DID NOT FIRE - do not trust anything below this line")
except LeakageError:
    tick("guard negative test passed (PV1MATH rejected)")

tick("Cell 2 OK")


## Cell 3 — School-grouped split and the model

Gradient boosting, because the inner loop of the nested run selected it in 728 of 750
outer folds. One school-grouped split is used, not the full nested design: the estimates
in the paper come from the nested run, and this notebook explains a fitted model rather
than re-estimating performance. The fold AUC is printed only as a sanity check against
the nested estimate.

In [ ]:

TASKS = [("low_vs_high","Low","High"), ("low_vs_medium","Low","Medium"),
         ("medium_vs_high","Medium","High")]
if not RUN_ALL_TASKS:
    TASKS = TASKS[:1]

MODEL_KW = dict(n_estimators=200, max_depth=3, learning_rate=0.1,
                subsample=1.0, random_state=RANDOM_SEED)

FITTED = {}
for name, neg, pos in TASKS:
    m = df["_cat"].isin([neg, pos]).to_numpy()
    Xt, gt = X_all[m], groups[m]
    yt = (df.loc[m, "_cat"] == pos).astype(int).to_numpy()
    wt = weights[m]

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    tr, te = next(sgkf.split(Xt, yt, groups=gt))
    assert not (set(gt[tr]) & set(gt[te])), "school appears on both sides of the split"

    clf = GradientBoostingClassifier(**MODEL_KW).fit(Xt.iloc[tr], yt[tr])
    p   = clf.predict_proba(Xt.iloc[te])[:, 1]
    auc = roc_auc_score(yt[te], p)

    FITTED[name] = dict(clf=clf, X_tr=Xt.iloc[tr], X_te=Xt.iloc[te],
                        y_te=yt[te], p_te=p, w_te=wt[te], auc=auc,
                        n_tr=len(tr), n_te=len(te),
                        schools_tr=len(set(gt[tr])), schools_te=len(set(gt[te])))
    tick(f"{name:16s} fold AUC {auc:.4f}   train {len(tr):,} / test {len(te):,}  "
         f"schools {len(set(gt[tr]))}/{len(set(gt[te]))}")

print("\nNested-CV reference (results/aggregate_summary.csv): "
      "Low vs. High 0.8850 | Low vs. Medium 0.7520 | Medium vs. High 0.6963")
print("A single fold will not reproduce these exactly; a gap beyond about 0.03 is worth checking.")
tick("Cell 3 OK")


## Cell 4 — Global SHAP

Aggregated over a stratified sample of test instances, with a bootstrap interval on each
feature's mean absolute SHAP value. Background and output scale are fixed here and written
to `explainer_config.csv` in cell 7.

**Output scale is the log-odds margin**, which is what `TreeExplainer` returns for a
gradient-boosting classifier. It is stated because SHAP values on the probability scale and
on the margin scale are not comparable, and the submitted version compared across
explainers whose scales differed.

In [ ]:

rng = np.random.default_rng(RANDOM_SEED)
GLOBAL = {}

for name, F in FITTED.items():
    Xte, yte = F["X_te"], F["y_te"]
    n = min(N_GLOBAL_SHAP, len(Xte))
    idx = np.concatenate([                       # stratified on the true label
        rng.choice(np.where(yte == c)[0],
                   size=max(1, int(round(n * (yte == c).mean()))), replace=False)
        for c in (0, 1)])
    Xs = Xte.iloc[idx]

    try:
        expl = shap.TreeExplainer(
            F["clf"],
            data=None,                            # interventional not needed for margin
            feature_perturbation="tree_path_dependent",
            model_output="raw",                   # log-odds margin
        )
    except TypeError:                             # older/newer shap signatures
        expl = shap.TreeExplainer(F["clf"])
        tick("  (shap fell back to the default TreeExplainer signature; "
             "scale is still the log-odds margin for this model)")
    sv = expl.shap_values(Xs, check_additivity=False)
    sv = sv[1] if isinstance(sv, list) else sv
    if sv.ndim == 3:
        sv = sv[:, :, 1]

    absmean = np.abs(sv).mean(axis=0)
    boot = np.empty((N_SHAP_BOOTSTRAP, sv.shape[1]))
    for b in range(N_SHAP_BOOTSTRAP):
        boot[b] = np.abs(sv[rng.integers(0, len(sv), len(sv))]).mean(axis=0)
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)

    signed = sv.mean(axis=0)                      # direction, reported separately
    g = (pd.DataFrame({"feature": Xs.columns, "mean_abs_shap": absmean,
                       "ci_low": lo, "ci_high": hi, "mean_signed_shap": signed,
                       "n_instances": len(Xs)})
           .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
    g["rank"] = np.arange(1, len(g) + 1)
    assert (g["mean_abs_shap"] >= 0).all(), "mean |SHAP| cannot be negative"

    GLOBAL[name] = dict(shap_values=sv, X=Xs, table=g, explainer=expl)
    g.to_csv(os.path.join(OUT, f"global_shap_{name}.csv"), index=False)
    tick(f"{name:16s} global SHAP over {len(Xs):,} instances; top: " +
         ", ".join(g["feature"].head(3)))

tick("Cell 4 OK")


## Cell 5 — Local SHAP, quantile-stratified

The submitted version explained the argmax and the argmin of predicted probability and
discussed them as though they characterised the population. Instances here are drawn at
fixed quantiles of the predicted-probability distribution, and the rule is recorded in the
output file so it can be checked rather than inferred.

In [ ]:

QUANTILES = [0.05, 0.25, 0.50, 0.75, 0.95, 0.99][:N_LOCAL_SHAP]
LOCAL = {}

for name, F in FITTED.items():
    G = GLOBAL[name]
    p_sub = F["clf"].predict_proba(G["X"])[:, 1]
    rows, chosen = [], []
    for q in QUANTILES:
        i = int(np.argmin(np.abs(p_sub - np.quantile(p_sub, q))))
        chosen.append((q, i))
        contrib = G["shap_values"][i]
        order = np.argsort(-np.abs(contrib))[:15]
        for r, j in enumerate(order, 1):
            rows.append(dict(task=name, selection_rule=f"quantile_{q:g}_of_predicted_probability",
                             instance_row=int(G["X"].index[i]), predicted_probability=float(p_sub[i]),
                             rank=r, feature=G["X"].columns[j],
                             feature_value=float(G["X"].iloc[i, j]),
                             shap_value=float(contrib[j])))
    t = pd.DataFrame(rows)
    t.to_csv(os.path.join(OUT, f"local_shap_{name}.csv"), index=False)
    LOCAL[name] = dict(table=t, chosen=chosen, p=p_sub)
    tick(f"{name:16s} local SHAP for {len(QUANTILES)} quantile-selected instances")

tick("Cell 5 OK")


## Cell 6 — LIME, repeated across seeds

LIME is stochastic and the submitted version set no seed. Each instance is explained
`N_LIME_SEEDS` times with different seeds; the standard deviation of every feature weight
and the mean Spearman correlation between the rankings of separate runs are reported.

This is the slowest cell — roughly 6 minutes per task.

In [ ]:

from scipy.stats import spearmanr
LIMES = {}

for name, F in FITTED.items():
    Xtr, G = F["X_tr"], GLOBAL[name]
    explainer_maker = lambda seed: lime.lime_tabular.LimeTabularExplainer(
        training_data=Xtr.to_numpy(),          # background: TRAINING data only
        feature_names=list(Xtr.columns),
        class_names=["negative", "positive"],
        discretize_continuous=True,
        mode="classification",
        random_state=seed,
    )
    predict = lambda a: F["clf"].predict_proba(a)

    rows = []
    for q, i in LOCAL[name]["chosen"]:
        x = G["X"].iloc[i].to_numpy()
        # W[feature] is a length-N_LIME_SEEDS vector indexed BY SEED, zero where that
        # seed's explanation did not name the feature. Appending to a list instead would
        # misalign the seeds and inflate the apparent agreement between runs.
        W = {}
        named = {}
        for s in range(N_LIME_SEEDS):
            exp = explainer_maker(RANDOM_SEED + s).explain_instance(
                x, predict, num_features=15, num_samples=N_LIME_SAMPLES)
            for fid, w in exp.as_map()[1]:
                f = Xtr.columns[fid]
                W.setdefault(f, np.zeros(N_LIME_SEEDS))[s] = w
                named.setdefault(f, []).append(s)
        # Rank correlation between runs, over the UNION of features named by any run.
        # Restricting to features named by every run silently discards the instability
        # being measured -- with 30 seeds that set is often empty. A feature a run did
        # not name carries weight 0 in that run's explanation, which is what is recorded.
        union = sorted(W)
        M = np.vstack([W[f] for f in union])          # features x seeds
        rho = np.nan
        if len(union) >= 3:
            rr = [spearmanr(M[:, a], M[:, b]).statistic
                  for a in range(N_LIME_SEEDS) for b in range(a + 1, N_LIME_SEEDS)]
            rr = [v for v in rr if np.isfinite(v)]
            rho = float(np.mean(rr)) if rr else np.nan
        for f in union:
            v = W[f]
            rows.append(dict(task=name, quantile=q, instance_row=int(G["X"].index[i]),
                             feature=f, n_seeds=N_LIME_SEEDS,
                             n_seeds_naming_it=len(named[f]),
                             mean_weight=float(v.mean()),
                             sd_weight=float(v.std(ddof=1)),
                             selection_frequency=len(named[f]) / N_LIME_SEEDS,
                             mean_pairwise_rank_correlation=rho))
        tick(f"  {name} q={q:g} done ({len(union)} distinct features across "
             f"{N_LIME_SEEDS} seeds; rank corr {rho:.3f})")

    t = pd.DataFrame(rows)
    t.to_csv(os.path.join(OUT, f"lime_stability_{name}.csv"), index=False)
    LIMES[name] = t
    tick(f"{name:16s} LIME stability written; mean rank correlation "
         f"{t['mean_pairwise_rank_correlation'].mean():.3f}")

tick("Cell 6 OK")


## Cell 7 — Codebook, configuration table, figures

The item wording is read from the SPSS metadata rather than written from memory, with a
small hand-verified override block for the four items the submitted manuscript
misidentified. `require_documented` raises if any interpreted item lacks a source, so an
undocumented gloss cannot reach a figure.

In [ ]:

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- codebook -------------------------------------------------------------
OVERRIDES = {   # hand-verified against PISA 2018 Student Questionnaire (CY7_201709_QST_MS_STQ_CBA)
 "ST012Q01TA": "In your home: Televisions (count)",
 "ST013Q01TA": "How many books are there in your home?",
 "ST004D01T" : "Student gender (standardised)",
 "ST166Q03HA": "Phishing e-mail scenario: click the link and fill out the form as soon as possible",
}
labels = {}
if os.path.exists(SAV):
    import pyreadstat
    _, meta = pyreadstat.read_sav(SAV, metadataonly=True)
    labels = dict(meta.column_names_to_labels or {})
    tick(f"SPSS metadata read: {len(labels):,} labelled columns")
else:
    tick("SPSS file absent; codebook falls back to the verified overrides only")

rows = []
for v in ALLOWLIST:
    if v in OVERRIDES:
        src, lbl = "hand-verified override", OVERRIDES[v]
        if v in labels and labels[v].strip():
            src += f" (SPSS metadata says: {labels[v].strip()})"
    elif v in labels and labels[v].strip():
        src, lbl = "SPSS metadata", labels[v].strip()
    else:
        src, lbl = "UNDOCUMENTED", ""
    rows.append(dict(item=v, label=lbl, source=src))
cb = pd.DataFrame(rows)
cb.to_csv(os.path.join(OUT, "codebook_resolved.csv"), index=False)

UNRESOLVED = cb.loc[cb["source"] == "UNDOCUMENTED", "item"].tolist()
if UNRESOLVED:
    print("\n" + "!" * 78)
    print("UNRESOLVED ITEMS -- no official wording available for:")
    print("  " + ", ".join(UNRESOLVED))
    print("Figures below use the bare item codes. Editor comment 7 requires a documented")
    print("description for every INTERPRETED item, so before any of these is discussed in")
    print("the manuscript, either place CY07_MSU_STU_QQQ.sav at")
    print(f"  {SAV}")
    print("and re-run this cell, or add the wording to OVERRIDES from the official")
    print("instrument (CY7_201709_QST_MS_STQ_CBA). Do NOT gloss an item from its code.")
    print("!" * 78 + "\n")

def require_documented(items):
    """Raise only for items that are actually interpreted in the manuscript."""
    bad = sorted(set(items) & set(UNRESOLVED))
    if bad:
        raise RuntimeError(
            f"these items are interpreted but have no documented source: {bad}. "
            "See the UNRESOLVED ITEMS notice above.")

# ---- explainer configuration ---------------------------------------------
cfg = pd.DataFrame([
 dict(explainer="SHAP TreeExplainer (global)", model="GradientBoostingClassifier",
      output_scale="log-odds margin (model_output='raw')",
      background="tree_path_dependent; no external background set",
      instances=f"{N_GLOBAL_SHAP} stratified on the true label",
      uncertainty=f"{N_SHAP_BOOTSTRAP}-resample bootstrap over instances", seed=RANDOM_SEED),
 dict(explainer="SHAP TreeExplainer (local)", model="GradientBoostingClassifier",
      output_scale="log-odds margin (model_output='raw')",
      background="tree_path_dependent; no external background set",
      instances=f"{len(QUANTILES)} at fixed quantiles of predicted probability",
      uncertainty="not applicable (single instance)", seed=RANDOM_SEED),
 dict(explainer="LIME LimeTabularExplainer", model="GradientBoostingClassifier",
      output_scale="predicted probability of the positive class",
      background="training fold only; discretize_continuous=True",
      instances=f"{len(QUANTILES)} instances x {N_LIME_SEEDS} seeds x {N_LIME_SAMPLES} perturbations",
      uncertainty="SD of each weight and mean pairwise Spearman correlation across seeds",
      seed=f"{RANDOM_SEED}..{RANDOM_SEED+N_LIME_SEEDS-1}"),
])
cfg.to_csv(os.path.join(OUT, "explainer_config.csv"), index=False)

# ---- figures --------------------------------------------------------------
for name in FITTED:
    g = GLOBAL[name]["table"].head(15).iloc[::-1]
    lab = {r.item: r.label[:52] for r in cb.itertuples() if r.label}
    y = np.arange(len(g))
    fig, ax = plt.subplots(figsize=(7.6, 5.4))
    ax.barh(y, g["mean_abs_shap"],
            xerr=[g["mean_abs_shap"] - g["ci_low"], g["ci_high"] - g["mean_abs_shap"]],
            color="#3b6ea5", ecolor="#888", capsize=2)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{f}\n{lab.get(f,'')}" if f in lab else f for f in g["feature"]], fontsize=7)
    ax.set_xlabel("mean |SHAP| (log-odds margin), 95% bootstrap CI")
    ax.set_title(f"Global SHAP — {name.replace('_',' ')}", fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, f"fig_global_shap_{name}.pdf")); plt.close(fig)

    t = LIMES[name]
    fig, ax = plt.subplots(figsize=(7.0, 4.6))
    ax.scatter(t["mean_weight"].abs(), t["sd_weight"], s=14, alpha=.6, color="#3b6ea5")
    lim = max(t["mean_weight"].abs().max(), t["sd_weight"].max()) * 1.05
    ax.plot([0, lim], [0, lim], ls="--", lw=.8, color="#aaa")
    ax.set_xlabel("|mean LIME weight| across seeds"); ax.set_ylabel("SD across seeds")
    ax.set_title(f"LIME stability — {name.replace('_',' ')} "
                 f"({N_LIME_SEEDS} seeds; points above the line are noise-dominated)", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, f"fig_lime_stability_{name}.pdf")); plt.close(fig)
    tick(f"{name:16s} figures written")

# Enforce the documentation rule on exactly the items the manuscript will interpret:
# the top 10 by global SHAP in each task. Everything above is already written to disk,
# so a failure here costs nothing but tells you what still needs a source.
INTERPRETED = sorted({f for name in FITTED
                        for f in GLOBAL[name]["table"]["feature"].head(10)
                        if f in set(ALLOWLIST)})
try:
    require_documented(INTERPRETED)
    tick(f"all {len(INTERPRETED)} interpreted items have a documented source")
except RuntimeError as e:
    print(f"\nWARNING (results are written; the manuscript claim is what is blocked)\n{e}\n")

manifest = dict(environment=ENV, model=MODEL_KW, allowlist=ALLOWLIST,
                unresolved_items=UNRESOLVED, interpreted_items=INTERPRETED,
                fold_auc={k: float(v["auc"]) for k, v in FITTED.items()},
                n_train={k: v["n_tr"] for k, v in FITTED.items()},
                n_test={k: v["n_te"] for k, v in FITTED.items()},
                files=sorted(os.listdir(OUT)))
with open(os.path.join(OUT, "manifest_explainability.json"), "w") as f:
    json.dump(manifest, f, indent=1)

print("\n".join(sorted(os.listdir(OUT))))
tick("Cell 7 OK — send Dr. Yazan the results/explainability/ folder")
